In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.common.qc import run_session_synapse_qc
from vip_slap2_analysis.behavior.preprocess import process_behavior_session
from vip_slap2_analysis.glutamate.extraction import process_glutamate_extraction
from vip_slap2_analysis.calcium.qc import run_calcium_qc
from vip_slap2_analysis.calcium.extraction import process_calcium_extraction

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
target_mice = [
#     803496,
    804730,804733,810196,
    809047,803121,
    826033,838410,834788
]

In [ ]:
registry = VIPSessionRegistry.from_basepath(
    r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics'
)

process_df = registry.sessions(
    subject_ids=target_mice,
    exclude_session_types=["expression_check","volume_imaging"],
    paradigms=["change_detection_passive"],
)

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

In [ ]:
datapath = r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033\826033_2026-02-21_09-23-34\analysis\derived\calcium\calcium_sequence_dff.npz"

In [ ]:
data = np.load(datapath,allow_pickle=True)['data'][0]

In [ ]:
dmd = 2
im_names = list(data[f'DMD{dmd}']['image_identity'].keys())

In [ ]:
roi = 2

soma_data =[data[f'DMD{dmd}']['image_identity'][im]['repeated']['mean'][:,roi,:] for im in im_names]

In [ ]:
im_colors = [
    '#c5cae9', '#ffcdd2', '#c8e6c9', '#ffe0b2',
    '#e1bee7', '#d7ccc8',
    '#9fd3f2']

In [ ]:
fs = 200
flash_start = 50 / fs      # 0.25 s
flash_dur = 50 / fs        # 0.25 s
gray_dur = 100 / fs        # 0.5 s
cycle_dur = flash_dur + gray_dur

for i,resp in enumerate(soma_data):
    shape = resp.shape
    
    n_flashes = shape[0]
    
    concat_traces = resp.reshape(shape[0] * shape[1])
    
    time  = np.linspace(0,len(concat_traces)/fs,len(concat_traces))
    
    fig,ax=plt.subplots()
    rolling = pd.DataFrame(concat_traces).rolling(5,min_periods=1).mean()
    ax.plot(time,rolling)
    
    for ii in range(n_flashes):
        start = flash_start + ii * cycle_dur
        end = start + flash_dur
        ax.axvspan(start, end, alpha=0.4, color=im_colors[i])

In [ ]:
data[f'DMD{dmd}']['image_identity']

In [ ]:
np.linspace(50,shape[0]*(50+150),shape[0])